# N0 — Data preparation and verification

Project 2. Produces the immutable artifacts every later notebook depends on.

**Outputs to Drive:**
- All six Shadow-ArUco viewpoints, MD5-verified
- Flying-ArUco v2 base dataset
- Dictionary verification across all six viewpoints
- Viewpoint-level splits with one viewpoint held out and untouched until final numbers
- COCO JSON + YOLO txt converters
- Frozen degradation seed table (`zlib.crc32`, not `hash()`)
- `manifest.json` with file hashes and environment versions

**Established in the dictionary investigation:** Shadow-ArUco and Flying-ArUco v2 use
**ARUCO_MIP_36h12**, not any `cv2.aruco` predefined dictionary. 250 markers, 6x6 bits,
minimum Hamming distance 12 → corrects 5 errors or 11 erasures.

Every stage asserts. If an assertion fires, fix the cause — do not accept an auto-generated
substitute, and do not let an assistant rewrite a data-loading cell.

## 1. Environment
Run, then **restart the runtime**, then continue from Cell 2.

In [ ]:
!pip uninstall -y opencv-python opencv-python-headless opencv-contrib-python opencv-contrib-python-headless -q
!pip install -q opencv-contrib-python==4.10.0.84 tqdm
print("\n>>> RESTART THE RUNTIME, then run Cell 2 onward.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 MB 10.9 MB/s eta 0:00:00

>>> RESTART THE RUNTIME, then run Cell 2 onward.


This cell first uninstalls several versions of OpenCV to ensure a clean environment, then installs a specific version (`opencv-contrib-python==4.10.0.84`) along with `tqdm` for progress bars. Finally, it prints a message instructing the user to restart the runtime before proceeding.

## 2. Paths, versions, environment log

In [ ]:
import os, sys, json, glob, hashlib, subprocess, zlib, platform, random
import numpy as np, cv2
from datetime import datetime, timezone

from google.colab import drive
drive.mount('/content/drive')

BASE   = '/content/drive/MyDrive/aruco_p2'
CACHE  = f'{BASE}/archives'
ART    = f'{BASE}/artifacts'
WORK   = '/content/work'
for d in (BASE, CACHE, ART, WORK):
    os.makedirs(d, exist_ok=True)

ENV = {
    'utc':        datetime.now(timezone.utc).isoformat(),
    'python':     sys.version.split()[0],
    'opencv':     cv2.__version__,
    'numpy':      np.__version__,
    'platform':   platform.platform(),
}
try:
    ENV['gpu'] = subprocess.check_output(
        ['nvidia-smi','--query-gpu=name','--format=csv,noheader']).decode().strip()
except Exception:
    ENV['gpu'] = 'none'

for k, v in ENV.items():
    print(f'{k:10s}: {v}')
if not cv2.__version__.startswith('4.10'):
    print('\nWARNING: expected OpenCV 4.10.0.84 — re-run Cell 1 and RESTART')

Mounted at /content/drive
utc       : 2026-09-05T23:34:28.901741+00:00
python    : 3.13.15
opencv    : 4.10.0
numpy     : 2.1.3
platform  : Linux-6.6.122+-x86_64-with-glibc2.35
gpu       : none


This cell sets up the environment. It imports necessary libraries like `os`, `sys`, `json`, `glob`, `hashlib`, `subprocess`, `zlib`, `platform`, `random`, `numpy`, `cv2`, and `datetime`. It then mounts Google Drive, defines paths for `BASE`, `CACHE`, `ART`, and `WORK` directories, and creates them if they don't exist. It also collects environment details such as UTC timestamp, Python version, OpenCV version, NumPy version, platform, and GPU information, printing them to the console. A warning is issued if the OpenCV version is not the expected `4.10.0.84`.

## 3. Shadow-ArUco — all six viewpoints

MD5s taken from the Zenodo record. ~4.6 GB total, cached on Drive so this is a one-time cost.
Verification before extraction is what prevents a truncated archive being analysed as complete.

In [ ]:
SHADOW_MD5 = {
    'video_1.tar.gz': '1c8bd58fb067a453487dfff783416674',
    'video_2.tar.gz': '1df7a6a568db96caf4e205a26050598f',
    'video_3.tar.gz': 'a21caebb578b5760c0efdd30472298bf',
    'video_4.tar.gz': 'd1d82d6054a1e61a6998e0069d8626e7',
    'video_5.tar.gz': '3e8acd8de355941aac414ef8ad44eb38',
    'video_6.tar.gz': 'be62ecc5969faec0e1a466b69e7cdda8',
}
SHADOW_URL = 'https://zenodo.org/records/10791293/files/{}'

def md5sum(p, chunk=1 << 20):
    h = hashlib.md5()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()

def fetch(name, url, expect_md5=None):
    dst = f'{CACHE}/{name}'
    if os.path.exists(dst) and (expect_md5 is None or md5sum(dst) == expect_md5):
        return dst, 'cached'
    subprocess.run(['curl', '-fL', '-o', dst, url], check=True)
    if expect_md5:
        got = md5sum(dst)
        assert got == expect_md5, f'{name}: MD5 {got} != {expect_md5} — delete and retry'
    return dst, 'downloaded'

for name, m in SHADOW_MD5.items():
    _, how = fetch(name, SHADOW_URL.format(name), m)
    print(f'{name}: {how}')

SHADOW = f'{WORK}/shadow'
os.makedirs(SHADOW, exist_ok=True)
for name in SHADOW_MD5:
    vid = name.replace('.tar.gz', '')
    if not os.path.isdir(f'{SHADOW}/{vid}'):
        subprocess.run(['tar', '-xzf', f'{CACHE}/{name}', '-C', SHADOW], check=True)

counts = {}
for name in SHADOW_MD5:
    vid = name.replace('.tar.gz', '')
    counts[vid] = len(glob.glob(f'{SHADOW}/{vid}/*.png'))
    print(f'  {vid}: {counts[vid]} frames')

total = sum(counts.values())
print(f'\ntotal frames: {total}')
assert total == 8652, f'expected 8652 frames (paper), got {total}'
print('all six viewpoints verified')

video_1.tar.gz: downloaded
video_2.tar.gz: downloaded
video_3.tar.gz: downloaded
video_4.tar.gz: downloaded
video_5.tar.gz: downloaded
video_6.tar.gz: downloaded
  video_1: 1433 frames
  video_2: 1378 frames
  video_3: 1362 frames
  video_4: 1417 frames
  video_5: 1334 frames
  video_6: 1728 frames

total frames: 8652
all six viewpoints verified


This cell handles the download and verification of the Shadow-ArUco dataset. It defines a dictionary `SHADOW_MD5` with MD5 checksums for six `.tar.gz` video archives and the base URL `SHADOW_URL`. It includes two helper functions: `md5sum` to calculate the MD5 hash of a file, and `fetch` to download a file from a URL, cache it, and verify its MD5 checksum. The cell then iterates through the `SHADOW_MD5` dictionary to download each video archive, verifying its integrity. After downloading, it extracts each `.tar.gz` file into the `SHADOW` directory. Finally, it counts the number of frames (PNG files) in each extracted video directory, sums them up, and asserts that the total number of frames is `8652`, confirming the integrity of the dataset.

## 4. Flying-ArUco v2 base dataset

2500 images with per-image JSON giving corner coordinates, ID, and rotation. This is the
`annotations_dir` that `build_regression.py` needs.

No published MD5, so verification is by content: image/JSON counts and schema.

In [ ]:
FA_NAME = 'flyingarucov2.tar.gz'
FA_URL  = f'https://zenodo.org/records/14053985/files/{FA_NAME}'
try:
    _, how = fetch(FA_NAME, FA_URL)
    print(f'{FA_NAME}: {how}')
except subprocess.CalledProcessError:
    raise RuntimeError(
        f'Download failed. Open https://zenodo.org/records/14053985 and check the exact '
        f'filename, then set FA_NAME accordingly.')

FLYING = f'{WORK}/flyingarucov2'
if not os.path.isdir(FLYING):
    os.makedirs(FLYING, exist_ok=True)
    subprocess.run(['tar', '-xzf', f'{CACHE}/{FA_NAME}', '-C', FLYING], check=True)

fa_imgs = sorted(glob.glob(f'{FLYING}/**/*.jpg', recursive=True))
fa_json = sorted(glob.glob(f'{FLYING}/**/*.json', recursive=True))
print(f'images: {len(fa_imgs)}   json: {len(fa_json)}')
assert len(fa_imgs) >= 2400, f'expected ~2500 images, got {len(fa_imgs)}'

with open(fa_json[0]) as f:
    s = json.load(f)
print('\nsample marker:', json.dumps(s['markers'][0], indent=1)[:260])
assert 'markers' in s and 'corners' in s['markers'][0] and 'id' in s['markers'][0]
HAS_ROT = 'rot' in s['markers'][0] or 'rotation' in s['markers'][0]
print(f'\nrotation field present: {HAS_ROT}   (needed by build_regression.py)')

flyingarucov2.tar.gz: downloaded
images: 2500   json: 2500

sample marker: {
 "id": 29,
 "corners": [
  [
   481.7849807739258,
   261.83265686035156
  ],
  [
   516.1624603271484,
   203.50667572021484
  ],
  [
   451.4819869995117,
   169.94456100463867
  ],
  [
   425.1454734802246,
   229.83663177490234
  ]
 ],
 "rot": 2
}

rotation field present: True   (needed by build_regression.py)


This cell downloads and verifies the Flying-ArUco v2 base dataset. It defines the archive name `FA_NAME` and its `FA_URL`. It attempts to fetch the archive using the `fetch` function and handles potential download failures. After successful download, it extracts the archive into the `FLYING` directory. It then counts the number of `.jpg` image files and `.json` annotation files, asserting that the image count is approximately 2500. Finally, it loads a sample JSON annotation file to print a sample marker's data and checks for the presence of a 'rot' or 'rotation' field, which is needed by downstream scripts.

## 5. Reference implementation

Clone the authors' repo. `build_detection.py`, `augment_dataset.py` and `build_regression.py`
are OpenCV/numpy only — they run on modern Python. Only the TensorFlow training scripts need
the Python 3.9 environment, and those are replaced by the PyTorch reimplementation in N1.

In [ ]:
REPO = f'{WORK}/deeparuco'
if not os.path.isdir(REPO):
    subprocess.run(['git','clone','--depth','1',
                    'https://github.com/AVAuco/deeparuco.git', REPO], check=True)

sys.path.insert(0, REPO)
from impl.aruco import MIP36h12, id_to_bits

print('MIP36h12 entries :', len(MIP36h12), '(250 markers + 2 negatives)')
bits = np.array([np.asarray(id_to_bits(i)).reshape(-1) for i in range(250)])
dmin = min(int(np.sum(bits[a] != bits[b])) for a in range(120) for b in range(a+1, 120))
print('min Hamming      :', dmin)
assert dmin == 12

# Reference decoder spec, transcribed from impl/architectures.py -> simple_decoder()
DECODER_SPEC = {
    'input': [32, 32, 1],
    'layers': [
        'Conv2D(8, 5x5, valid) -> PReLU -> MaxPool2D',
        'Conv2D(4, 3x3, valid) -> PReLU -> MaxPool2D',
        'Conv2D(1, 1x1, valid, sigmoid)',
    ],
    'output': [6, 6, 1],
    'loss': 'mae', 'optimizer': 'adam', 'batch_size': 32,
    'epochs': 1000, 'early_stop_patience': 20,
    'reduce_lr_patience': 10, 'reduce_lr_factor': 0.5,
    'crop_size': 64, 'bbox_expand': 0.20, 'warp_size': 32,
    'corner_jitter': 0.05,
    'augment': '4 rotations + h/v flips (bits transformed to match) + lighting gradients',
}
print('\ndecoder spec captured for N1:')
print(json.dumps(DECODER_SPEC, indent=1))

MIP36h12 entries : 252 (250 markers + 2 negatives)
min Hamming      : 12

decoder spec captured for N1:
{
 "input": [
  32,
  32,
  1
 ],
 "layers": [
  "Conv2D(8, 5x5, valid) -> PReLU -> MaxPool2D",
  "Conv2D(4, 3x3, valid) -> PReLU -> MaxPool2D",
  "Conv2D(1, 1x1, valid, sigmoid)"
 ],
 "output": [
  6,
  6,
  1
 ],
 "loss": "mae",
 "optimizer": "adam",
 "batch_size": 32,
 "epochs": 1000,
 "early_stop_patience": 20,
 "reduce_lr_patience": 10,
 "reduce_lr_factor": 0.5,
 "crop_size": 64,
 "bbox_expand": 0.2,
 "warp_size": 32,
 "corner_jitter": 0.05,
 "augment": "4 rotations + h/v flips (bits transformed to match) + lighting gradients"
}


This cell clones the DeepArUco reference implementation repository from GitHub into the `REPO` directory. It then adds the repository's path to `sys.path` so that its modules can be imported. It imports `MIP36h12` and `id_to_bits` from `impl.aruco`. It prints the number of entries in `MIP36h12` and calculates the minimum Hamming distance between the first 120 marker bits, asserting that it is `12`. Finally, it captures the decoder specification from `impl/architectures.py` as `DECODER_SPEC` and prints it, noting it's for use in a subsequent notebook (N1).

## 6. Dictionary verification across all six viewpoints

video_1 was verified during the investigation. Confirm the other five before assuming the
whole dataset shares one dictionary. `DICT_6X6_250` runs as a control and must stay at zero.

In [ ]:
def mip36h12_dictionary(n=250, maxcorr=5):
    probe = cv2.aruco.Dictionary.getByteListFromBits(np.asarray(id_to_bits(0),dtype=np.uint8).reshape(6,6))
    bl = np.zeros((n, probe.shape[1], 4), dtype=np.uint8)
    for i in range(n):
        bl[i] = cv2.aruco.Dictionary.getByteListFromBits(
            np.asarray(id_to_bits(i), dtype=np.uint8).reshape(6, 6))
    return cv2.aruco.Dictionary(bl, 6, maxcorr)

params = cv2.aruco.DetectorParameters()
params.adaptiveThreshWinSizeMin, params.adaptiveThreshWinSizeMax = 3, 53
params.adaptiveThreshWinSizeStep = 4
params.minMarkerPerimeterRate = 0.01
params.polygonalApproxAccuracyRate = 0.06

det_mip = cv2.aruco.ArucoDetector(mip36h12_dictionary(), params)
det_ctl = cv2.aruco.ArucoDetector(
    cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_6X6_250), params)
clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))

def pairs_for(vid):
    out = []
    for f in sorted(glob.glob(f'{SHADOW}/{vid}/*.png')):
        j = f'{SHADOW}/{vid}/corrected_annotations/{os.path.splitext(os.path.basename(f))[0]}.json'
        if os.path.exists(j):
            out.append((f, j))
    return out

def get_corners(m):
    c = m['corners']
    if isinstance(c[0], dict):
        return np.array([[p['x'], p['y']] for p in c], dtype=np.float32)
    return np.asarray(c, dtype=np.float32).reshape(4, 2)

PAIRS, BASELINE = {}, {}
print(f"{'viewpoint':<12}{'frames':>8}{'IDs':>22}{'MIP36h12':>11}{'control':>9}")
print('-' * 63)
for vid in sorted(counts):
    P = pairs_for(vid); PAIRS[vid] = P
    ids_seen = set(); tp = ctl = tot = 0
    for f, j in P[::12]:
        g = clahe.apply(cv2.imread(f, cv2.IMREAD_GRAYSCALE))
        with open(j) as fp:
            gt = [int(m['id']) for m in json.load(fp)['markers']]
        ids_seen.update(gt); tot += len(gt)
        _, ids, _ = det_mip.detectMarkers(g)
        if ids is not None:
            tp += sum(int(i) in gt for i in ids.flatten().tolist())
        _, cid, _ = det_ctl.detectMarkers(g)
        if cid is not None:
            ctl += sum(int(i) in gt for i in cid.flatten().tolist())
    r = tp / max(tot, 1)
    BASELINE[vid] = {'recall_clahe': r, 'n_markers': tot, 'ids': sorted(ids_seen)}
    rng = f'{min(ids_seen)}-{max(ids_seen)} ({len(ids_seen)})'
    print(f'{vid:<12}{len(P):>8}{rng:>22}{r:>11.4f}{ctl/max(tot,1):>9.4f}')
    assert ctl == 0, f'{vid}: control dictionary detected markers — investigate'
    assert r > 0,   f'{vid}: MIP36h12 recall zero — investigate before proceeding'

print('\nARUCO_MIP_36h12 confirmed on all six viewpoints; control zero throughout.')

viewpoint     frames                   IDs   MIP36h12  control
---------------------------------------------------------------
video_1         1433          238-249 (12)     0.6354   0.0000
video_2         1378           238-249 (9)     0.6502   0.0000
video_3         1362          238-249 (12)     0.7178   0.0000
video_4         1417          238-249 (11)     0.5959   0.0000
video_5         1334           238-249 (9)     0.6448   0.0000
video_6         1728          238-249 (12)     0.6244   0.0000

ARUCO_MIP_36h12 confirmed on all six viewpoints; control zero throughout.


This cell verifies the Aruco dictionary across all six Shadow-ArUco viewpoints. It defines a custom `mip36h12_dictionary` function to create an Aruco dictionary from the `id_to_bits` mapping. It sets up `DetectorParameters` and initializes two `ArucoDetector` instances: `det_mip` for the custom dictionary and `det_ctl` for a standard `DICT_6X6_250` as a control. It also initializes `CLAHE` for image preprocessing. The `pairs_for` function collects image-annotation pairs for a given video. The main loop iterates through each video viewpoint, processing a subset of frames (every 12th frame). For each frame, it reads the image, applies CLAHE, loads ground truth marker IDs, and attempts to detect markers using both `det_mip` and `det_ctl`. It calculates the recall for `det_mip` and asserts that the control detector `det_ctl` finds zero markers, confirming that `ARUCO_MIP_36h12` is the correct dictionary for the dataset.

## 7. Viewpoint-level splits

Never split Shadow-ArUco by frame — it is video from six fixed cameras and adjacent frames are
near-duplicates. Random frame splits leak and inflate every number.

One viewpoint is held out entirely. Do not evaluate on it until the final numbers, so that
repeated N1/N2 iteration cannot tune against it.

In [ ]:
HELDOUT = 'video_6'
DEV     = [v for v in sorted(counts) if v != HELDOUT]

splits = {
    'heldout': [HELDOUT],
    'dev':     DEV,
    'policy':  ('Shadow-ArUco is evaluation-only; training uses Flying-ArUco v2. '
                'Splits are by viewpoint sequence, never by frame. The heldout viewpoint '
                'is untouched until final reporting.'),
    'benchmark_stride': 6,
}
with open(f'{ART}/splits.json', 'w') as f:
    json.dump(splits, f, indent=1)

print('dev viewpoints :', DEV)
print('held out       :', HELDOUT, f'({counts[HELDOUT]} frames)')
print('\nbenchmark subsampling: every 6th frame')
for v in sorted(counts):
    print(f'  {v}: {counts[v]:5d} -> {len(range(0, counts[v], 6)):5d}')

dev viewpoints : ['video_1', 'video_2', 'video_3', 'video_4', 'video_5']
held out       : video_6 (1728 frames)

benchmark subsampling: every 6th frame
  video_1:  1433 ->   239
  video_2:  1378 ->   230
  video_3:  1362 ->   227
  video_4:  1417 ->   237
  video_5:  1334 ->   223
  video_6:  1728 ->   288


This cell defines how the dataset is split for development and evaluation. It designates `video_6` as the `HELDOUT` viewpoint, which is kept separate for final evaluation to prevent overfitting during development. The `DEV` set consists of the remaining viewpoints. It then creates a `splits` dictionary containing the heldout and development viewpoints, a policy statement on splitting strategy, and a `benchmark_stride` for subsampling. This `splits` information is saved as `splits.json` in the `ART` directory. Finally, it prints the development and heldout viewpoints, along with the number of frames and the subsampled frame count for each video based on the `benchmark_stride`.

## 8. Converters

`to_coco` for DETR / RT-DETR, `to_yolo` for Ultralytics. Both derive an axis-aligned bbox from
the four corners, and **both preserve the raw corners**, which the rectification and decoding
stages need. Round-trip tested below.

In [ ]:
def corners_to_bbox(c):
    x, y = c[:, 0], c[:, 1]
    return float(x.min()), float(y.min()), float(x.max()-x.min()), float(y.max()-y.min())

def to_yolo(corners, W, H):
    x, y, w, h = corners_to_bbox(corners)
    return (np.clip((x+w/2)/W,0,1), np.clip((y+h/2)/H,0,1),
            np.clip(w/W,0,1), np.clip(h/H,0,1))

def to_coco(pair_list, out_path, category='marker'):
    images, annots, aid = [], [], 1
    for iid, (f, j) in enumerate(pair_list, 1):
        H, W = cv2.imread(f, cv2.IMREAD_GRAYSCALE).shape
        images.append({'id': iid, 'file_name': os.path.basename(f),
                       'width': W, 'height': H, 'path': f})
        with open(j) as fp:
            for m in json.load(fp)['markers']:
                c = get_corners(m)
                x, y, w, h = corners_to_bbox(c)
                annots.append({'id': aid, 'image_id': iid, 'category_id': 1,
                               'bbox': [x, y, w, h], 'area': w*h, 'iscrowd': 0,
                               'marker_id': int(m['id']),
                               'corners': c.reshape(-1).tolist()})
                aid += 1
    doc = {'images': images, 'annotations': annots,
           'categories': [{'id': 1, 'name': category}]}
    with open(out_path, 'w') as f:
        json.dump(doc, f)
    return doc

demo = to_coco(PAIRS[DEV[0]][:40], f'{ART}/_convert_test.json')
print(f'COCO test: {len(demo["images"])} images, {len(demo["annotations"])} annotations')

a = demo['annotations'][0]
c = np.array(a['corners']).reshape(4, 2)
x, y, w, h = a['bbox']
assert abs(c[:,0].min()-x) < 1e-6 and abs(c[:,1].min()-y) < 1e-6
assert c[:,0].max() <= x+w+1e-6 and c[:,1].max() <= y+h+1e-6
print('bbox encloses corners, corners preserved: OK')

cx, cy, nw, nh = to_yolo(c, demo['images'][0]['width'], demo['images'][0]['height'])
assert all(0 <= v <= 1 for v in (cx, cy, nw, nh))
print(f'YOLO norm: cx={cx:.4f} cy={cy:.4f} w={nw:.4f} h={nh:.4f}  OK')
os.remove(f'{ART}/_convert_test.json')

COCO test: 40 images, 480 annotations
bbox encloses corners, corners preserved: OK
YOLO norm: cx=0.6641 cy=0.4110 w=0.1238 h=0.1422  OK


This cell defines converter functions for generating COCO JSON and YOLO text annotation formats from the Aruco marker data, preserving the raw corner coordinates. The `corners_to_bbox` function calculates an axis-aligned bounding box from given corner points. The `to_yolo` function converts corner coordinates and image dimensions into YOLO format (normalized center x, center y, width, height). The `to_coco` function takes a list of image-annotation pairs and an output path to generate a COCO JSON file, including image metadata and annotations with bounding box, area, marker ID, and raw corners. A demonstration is performed by converting a small subset of data, and assertions confirm that bounding boxes correctly enclose corners and YOLO normalization is correct. A temporary test file is created and then removed.

## 9. Frozen degradation seed table

Project 1's `bench.py` seeded with `hash((name, lvl, iid))`. Python randomises string hashing
per process, so stochastic operators were not reproducible across sessions — which is why the
whole benchmark had to run atomically and ultimately had to be regenerated.

`zlib.crc32` is stable across processes, machines and Python versions. Materialising the table
as a CSV artifact makes reproducibility auditable rather than asserted, and lets evaluation be
split across sessions and resumed after a crash.

In [ ]:
OPERATORS = ['gaussian_blur','motion_blur','gaussian_noise','jpeg',
             'brightness','contrast','downscale','occlusion']
LEVELS = [1, 2, 3, 4, 5, 6]

def seed_for(op, lvl, image_key):
    return zlib.crc32(f'{op}|{lvl}|{image_key}'.encode('utf-8'))

# stability proof: crc32 is identical across processes, hash() is not
probe = "gaussian_noise|3|video_1/000123"
sub = subprocess.run([sys.executable,'-c',
    f'import zlib;print(zlib.crc32({probe!r}.encode()));print(hash({probe!r}))'],
    capture_output=True, text=True)
crc_other, hash_other = sub.stdout.split()
print(f'crc32  this process: {zlib.crc32(probe.encode())}   other: {crc_other}')
print(f'hash() this process: {hash(probe)}   other: {hash_other}')
assert int(crc_other) == zlib.crc32(probe.encode()), 'crc32 must match across processes'
print('\ncrc32 stable across processes; hash() is not — this is the Project 1 bug.')

rows = ['operator,level,image_key,seed\n']
for vid in sorted(counts):
    for f, _ in PAIRS[vid][::6]:
        key = f'{vid}/{os.path.splitext(os.path.basename(f))[0]}'
        for op in OPERATORS:
            for lvl in LEVELS:
                rows.append(f'{op},{lvl},{key},{seed_for(op,lvl,key)}\n')
with open(f'{ART}/seed_table.csv', 'w') as f:
    f.writelines(rows)
print(f'\nseed table: {len(rows)-1} rows -> {ART}/seed_table.csv')

crc32  this process: 1024654211   other: 1024654211
hash() this process: -6809967962669065445   other: 2659149080796294239

crc32 stable across processes; hash() is not — this is the Project 1 bug.

seed table: 69312 rows -> /content/drive/MyDrive/aruco_p2/artifacts/seed_table.csv


This cell addresses a reproducibility issue from Project 1 by creating a frozen degradation seed table using `zlib.crc32` instead of `hash()`. It defines `OPERATORS` (types of degradation) and `LEVELS`. The `seed_for` function generates a seed using `zlib.crc32` for a given operator, level, and image key. It then includes a stability proof, showing that `zlib.crc32` produces identical results across different processes while `hash()` does not, highlighting the Project 1 bug. Finally, it iterates through all video frames (subsampled by `benchmark_stride`), all operators, and all levels to generate a comprehensive seed table, which is then saved as `seed_table.csv` in the `ART` directory. This ensures reproducible degradation across sessions.

## 10. Manifest

Hashes and environment recorded together. This is what the paper's reproducibility statement
cites, and what tells you months later exactly which data and versions produced a result.

In [ ]:
manifest = {
    'created': datetime.now(timezone.utc).isoformat(),
    'environment': ENV,
    'dictionary': {
        'name': 'ARUCO_MIP_36h12',
        'source': 'AVAuco/deeparuco impl/aruco.py',
        'n_markers': 250, 'bits': 36, 'grid': '6x6', 'border_cells': 1,
        'min_hamming_distance': 12,
        'max_error_correction': 5, 'max_erasure_correction': 11,
        'note': ('NOT a cv2.aruco predefined dictionary. The DeepArUco++ paper describes '
                 'markers as from "the ArUco DICT 6X6 250 dictionary", which denotes 6x6 bits '
                 'and 250 markers, not the OpenCV constant DICT_6X6_250. Every OpenCV '
                 'predefined dictionary yields zero detections on this data.'),
    },
    'shadow_aruco': {
        'doi': '10.5281/zenodo.10791293', 'license': 'CC-BY-4.0',
        'frames': counts, 'total_frames': total,
        'md5': SHADOW_MD5,
        'classical_baseline_clahe_stride12': {k: round(v['recall_clahe'], 4)
                                              for k, v in BASELINE.items()},
    },
    'flying_aruco_v2': {
        'doi': '10.5281/zenodo.14053985',
        'archive': FA_NAME, 'md5': md5sum(f'{CACHE}/{FA_NAME}'),
        'images': len(fa_imgs),
    },
    'splits': splits,
    'decoder_reference_spec': DECODER_SPEC,
    'seeding': {'function': 'zlib.crc32', 'format': 'operator|level|image_key',
                'operators': OPERATORS, 'levels': LEVELS,
                'replaces': 'hash() from Project 1 bench.py (process-randomised)'},
}
with open(f'{ART}/manifest.json', 'w') as f:
    json.dump(manifest, f, indent=1)

print(json.dumps({k: (v if not isinstance(v, dict) else '...')
                  for k, v in manifest.items()}, indent=1))
print(f'\nartifacts in {ART}:')
for p in sorted(glob.glob(f'{ART}/*')):
    print(f'  {os.path.basename(p):<20}{os.path.getsize(p)/1024:>10.1f} KB')
print('\nN0 complete. Next: N0.5 (weights smoke test), then N1 (decoder + headroom).')

{
 "created": "2026-09-05T23:53:18.757155+00:00",
 "environment": "...",
 "dictionary": "...",
 "shadow_aruco": "...",
 "flying_aruco_v2": "...",
 "splits": "...",
 "decoder_reference_spec": "...",
 "seeding": "..."
}

artifacts in /content/drive/MyDrive/aruco_p2/artifacts:
  manifest.json              3.0 KB
  seed_table.csv          2537.7 KB
  splits.json                0.3 KB

N0 complete. Next: N0.5 (weights smoke test), then N1 (decoder + headroom).


This cell creates a `manifest.json` file that records all critical information about the data, environment, and processing steps for reproducibility. It includes: creation timestamp, environmental details (`ENV`), dictionary specification (`ARUCO_MIP_36h12`), Shadow-ArUco dataset details (DOI, license, frame counts, MD5s, classical baseline recall), Flying-ArUco v2 details (DOI, archive name, MD5, image count), data splits, decoder reference specification, and seeding mechanism details. This `manifest` dictionary is then written to `manifest.json` in the `ART` directory. Finally, it prints a condensed version of the manifest and lists all generated artifacts in the `ART` directory with their sizes, indicating the completion of N0.